# Langchain Agents 
This notebook demonstrates how to create and use agents in Langchain, including the use of tools like DuckDuckGoSearch for real-time information retrieval.

At the end of this notebook, we can see how to create a ReAct agent from a prompt template and use it with a memory system to maintain chat history.



In [32]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq


In [33]:
# load environment variables from .env file
load_dotenv()

True

In [34]:
# Get llms models via the `langchain_groq` package
llm_model = ChatGroq(
    model="llama-3.3-70b-versatile", #llama-3.3-70b-versatile",
    temperature=.5,
    max_tokens=512,
)

## Langchain Tools

The [third-party tools](https://python.langchain.com/docs/integrations/tools/) from the langchain-community ecosytem allow us to extent the capabilities of our agents

In [35]:
# from langchain_community.tools import DuckDuckGoSearchRun
from ddgs import DDGS
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

`DuckDuckGoSearchRun` allows us to perform web searches and retrieve information in real-time, which can be very useful for agents that need to answer questions based on current data.


In [36]:
#api_wrapper = DuckDuckGoSearchAPIWrapper(region="en-en",max_results=10)
#search = DuckDuckGoSearchRun()#,api_wrapper=api_wrapper)

def ddg_search(query: str):
    with DDGS() as ddgs:
        return ddgs.text(query, max_results=5)
search = ddg_search

In [37]:
results = search("Prime Italian Minister")  

In [38]:
results

[{'title': 'Prime Minister of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Prime_Minister_of_Italy',
  'body': 'The Italian prime minister presided over a very unstable political system as in its first sixty years of existence (1861–1921) Italy changed its ...'},
 {'title': 'List of prime ministers of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/List_of_Prime_Ministers_of_Italy',
  'body': 'Top right: Benito Mussolini was the longest-serving prime minister in Italian history. ... was the longest-serving prime minister of the Italian ...'},
 {'title': 'Former Italian prime minister Silvio Berlusconi dies aged 86 -',
  'href': 'https://www.dailymotion.com/video/x8lp8iu',
  'body': 'Former Italian prime minister Silvio Berlusconi dies aged 86 ... Silvio Berlusconi: Former Italian prime minister dies aged 86'},
 {'title': "In White House meeting, Italian prime minister vows to 'make",
  'href': 'https://www.upi.com/Top_News/US/2025/04/17/italys-meloni-voes-t

In [39]:
# Define a tool for the DuckDuckGo search
from langchain.tools import Tool
search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Use this tool when you need to search for real-time information from DuckDuckGo."
)

In [40]:
from langchain.agents import initialize_agent, AgentType


In [41]:
# Initialize the agent with the DuckDuckGo search tool
agent = initialize_agent(
    tools=[search_tool],
    llm=llm_model,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=7,
    agent_name="DuckDuckGoSearchAgent",
    agent_description="An agent that can search the web using DuckDuckGoSearch and summarize findings clearly."
)

# Now we can use the agent to answer questions
results = agent.invoke("Who is the Italian Prime Minister?")  # Example query to the agent



> Entering new AgentExecutor chain...
To find the current Italian Prime Minister, I should search for real-time information. 

Action: DuckDuckGoSearch
Action Input: Italian Prime Minister

(I'll wait for the search results...)
Observation: [{'title': 'List of prime ministers of Italy - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/List_of_prime_ministers_of_Italy', 'body': 'Alcide De Gasperi is the only prime minister who has held this position both in the Kingdom of Italy and in the Republic of Italy. The prime minister is appointed by the President of the Republic and must receive a confidence vote by both houses of Parliament: the Chamber of Deputies and the Senate. [8]'}, {'title': 'Giorgia Meloni - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Giorgia_Meloni', 'body': 'Giorgia Meloni ( Italian : [ˈdʒordʒa meˈloːni]; born 15 January 1977) is an Italian politician who has served as Prime Minister of Italy since October 2022. She is the first woman to hold the office and

### Create a an Agent with DuckDuckGoSearch with Memory
In this section, we will create an agent that can search the web using DuckDuckGoSearch and maintain a memory of previous interactions. 

In [42]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder


In [43]:

# Build a chat prompt that includes conversation history so the agent can resolve references.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Answer the following questions as best you can. You have access to the following tools:
{tools}
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("assistant", "{agent_scratchpad}"),
])


The react_template instructs the laguage model to:
1. __Think__ about how to solve a user's question
2. __Use tools__ (actions) to gather mor information or perform actions
3. __Observe results__ and update the its thoughts
4. Eventually, reach a __Final Answer__

The following is the explanation of the `react_template` used to create the agent:

```python
react_template = """
'Answer the following questions as best you can. You have access to the following tools:

{tools}
```
+ Introduce the agent and tells it what tools it can use
+ {tools} is a placeholder for the tools that will be used by the agent

```text
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
```

This is the strict __format__ the LLM must follow to:

* Show its reasoning (Thought)

* Choose and execute a tool (Action)

* Feed it input (Action Input)

* Observe results (Observation)

* Loop until it gets to the final answer

```text
Begin!
Question: {input}
Thought:{agent_scratchpad}'
"""
```
+ `{input}` is the actual user question.

+ `{agent_scratchpad}` holds any intermediate steps already completed (e.g., previous Thoughts, Actions, Observations).

+ The agent continues from this point.

In [44]:
# prompt_template defined above with chat history support


In [45]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [46]:
# Simple session store to keep track of chat history per conversation
store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [47]:
# Create the agent 
agent = create_react_agent(llm_model, [search_tool], prompt_template)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True,
    handle_parsing_errors=True,
)


In [48]:
# Wrap the agent executor with message history
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)


In [49]:
# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "How many people live in italy?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run bcabc69a-50d0-4afa-82fd-4f77af55c2d4 not found for run db82d977-bc63-4b25-b382-22205ccb7002. Treating as a root run.




> Entering new AgentExecutor chain...
Question: How many people live in Italy?
Thought: To find the current population of Italy, I need to search for the most recent data available.
Action: DuckDuckGoSearch
Action Input: Italy population 2024[{'title': 'Demographics of Italy - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Demographics_of_Italy', 'body': 'The demography of Italy is monitored by the Italian National Institute of Statistics (Istat). At the beginning of 2024 , Italy had an estimated population of 58.9 million.[1] Its population ...'}, {'title': 'Italy Population (2026) - Worldometer', 'href': 'https://www.worldometers.info/world-population/italy-population/', 'body': 'The median age in Italy is 48.6 years. 1. U.N. Dept. of Economic and Social Affairs - Population Division. World Population Prospects: The 2024 Revision. (Medium-fertility variant).'}, {'title': 'Italy Population , 1911 – 2025 | CEIC Data', 'href': 'https://www.ceicdata.com/en/indicator/italy/populatio

{'input': 'How many people live in italy?',
 'chat_history': [],
 'output': 'The estimated population of Italy is approximately 58.9 million.'}

In [50]:
# Let's ask another question to see how the agent uses the chat history
agent_with_chat_history.invoke(
    {"input": "what is their national anthem called?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 0c7dd95a-5d6f-44f0-baaa-b4b8cac1a15c not found for run 8f15f2da-af1c-4985-916a-b9570f2b8247. Treating as a root run.




> Entering new AgentExecutor chain...
Thought: To find the name of Italy's national anthem, I should recall general knowledge or search for it.
The national anthem of Italy is called "Il Canto degli Italiani" (The Song of the Italians), also known as "Inno di Mameli" (Mameli's Hymn).Invalid Format: Missing 'Action:' after 'Thought: However I can correct that and provide the answer directly 
Final Answer: Il Canto degli Italiani

> Finished chain.


{'input': 'what is their national anthem called?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The estimated population of Italy is approximately 58.9 million.')],
 'output': 'Il Canto degli Italiani'}

### Exercise
Create or Select some tools from the [langchain-community tools](https://python.langchain.com/docs/integrations/tools/) and create a ReAct agent that can use them to solve certain tasks.

In [51]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain.tools import Tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import ChatPromptTemplate

# ----------------------------
# 1) LLM
# ----------------------------
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
    max_tokens=600,
)

# ----------------------------
# 2) Tools
# ----------------------------
from ddgs import DDGS

def ddg_search(query: str):
    with DDGS() as ddgs:
        return ddgs.text(query, max_results=5)

duckduckgo_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Real-time web lookup (current facts, names, headlines). Input: short search query."
)

# Wikipedia tool (requires: pip install wikipedia)
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=1200))
wikipedia_tool = Tool(
    name="Wikipedia",
    func=wiki.run,
    description="Background/encyclopedia explanations (definitions, history, concepts). Input: topic string."
)

# Safe calculator (no eval)
import ast, operator as op
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.Mod: op.mod, ast.FloorDiv: op.floordiv
}

def safe_calc(expr: str) -> str:
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
            return _ALLOWED_OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
            return _ALLOWED_OPS[type(node.op)](_eval(node.operand))
        raise ValueError("Unsupported expression.")
    return str(_eval(ast.parse(expr, mode="eval").body))

calculator_tool = Tool(
    name="Calculator",
    func=safe_calc,
    description="Arithmetic only. Input: expression like '15*4' or '128*0.15'."
)

tools = [duckduckgo_tool, wikipedia_tool, calculator_tool]

# ----------------------------
# 3) ReAct prompt (IMPORTANT: scratchpad as STRING placeholder)
# ----------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant with tools.\n"
     "Use tools when needed and keep answers tight.\n\n"
     "Tools:\n{tools}\n\n"
     "Format:\n"
     "Question: {input}\n"
     "Thought: ...\n"
     "Action: one of [{tool_names}]\n"
     "Action Input: ...\n"
     "Observation: ...\n"
     "... (repeat)\n"
     "Final Answer: ...\n"),
    ("human", "{input}"),
    ("assistant", "{agent_scratchpad}"),  # <-- THIS FIXES YOUR ERROR
])

# ----------------------------
# 4) Build agent + executor
# ----------------------------
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    handle_parsing_errors=True,
)

# ----------------------------
# 5) Tests (DO NOT pass agent_scratchpad yourself)
# ----------------------------
print("\n--- TEST 1 ---")
out1 = agent_executor.invoke({
    "input": "Who is the current Prime Minister of Italy, and how many letters are in their last name? Use tools."
})
print(out1["output"])

print("\n--- TEST 2 ---")
out2 = agent_executor.invoke({
    "input": "Explain the ReAct pattern in 3 bullets and then calculate 128*0.15."
})
print(out2["output"])


--- TEST 1 ---


> Entering new AgentExecutor chain...
Thought: To find the current Prime Minister of Italy, I should use a real-time web lookup.

Action: DuckDuckGoSearch
Action Input: current Prime Minister of Italy
[{'title': 'Giorgia Meloni - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Giorgia_Meloni', 'body': 'Giorgia Meloni (Italian: [ˈdʒordʒa meˈloːni]; born 15 January 1977) is an Italian politician who has served as Prime Minister of Italy since October 2022. She is the first woman to hold the office and the head of the third-longest government in the history of the Italian Republic.'}, {'title': 'Giorgia Meloni | Party, Policies, & Italy | Britannica', 'href': 'https://www.britannica.com/biography/Giorgia-Meloni', 'body': 'Feb 8, 2026 · Giorgia Meloni is a populist Italian politician who cofounded (2012) and leads (2014– ) the Brothers of Italy (Fratelli d’Italia), a party with neofascist roots. She is the first woman to serve as prime minister of Italy (2022– ).'}, {'

---

# 🧠 Notebook 2 — LangChain Agents (ReAct + Retrieval + Memory)

## What We Built

An upgraded ReAct agent that can:

- 🌐 Search the web in real time (DuckDuckGo)
- 🧠 Follow a strict reasoning format
- 💬 Maintain multi-turn conversation memory
- 🏗 Use modern LangChain agent APIs

This moves from simple tool demos → functional agent systems.

---

## Core Architecture

### 1️⃣ Retrieval + Reasoning

The agent now:

1. Thinks about the task  
2. Decides to search  
3. Observes search results  
4. Refines its answer  
5. Repeats if needed  

Tool definition:

```python
Tool(
    name="DuckDuckGoSearch",
    description="Use when real-time information is required."
)

Critical reminder:
The agent only understands the description, not the tool’s internal logic

2️⃣ Structured ReAct Prompt

We enforced strict reasoning format:
Question
Thought
Action
Action Input
Observation
...
Final Answer

Key placeholders:
	•	{tools}
	•	{tool_names}
	•	{input}
	•	{agent_scratchpad}
	•	chat_history

Prompt structure = behavioral control.

3️⃣ Memory Layer

Added session-based memory using:
	•	ChatMessageHistory
	•	RunnableWithMessageHistory
	•	session_id

Now the agent can resolve references:

Q1: How many people live in Italy?
Q2: What is their national anthem called?

“Italy” is remembered via chat history.

Memory transforms single-shot agents → conversational systems.

# 4 Modern Construction Pattern

Replaced deprecated initialize_agent with:

create_react_agent(...)
AgentExecutor(...)
RunnableWithMessageHistory(...)

Cleaner. Modular. Production-aligned.


# Key Takeaways
	•	Retrieval + reasoning enables dynamic knowledge.
	•	Tool descriptions determine routing reliability.
	•	Prompt formatting enforces disciplined thought loops.
	•	Memory enables contextual continuity.
	•	Agents are orchestration layers — not just LLM calls.

⸻

Positioning

Notebook 1 → Basic ReAct agent
Notebook 2 → Web-enabled + memory-enabled agent

Next layer:
	•	Multi-tool orchestration
	•	RAG systems
	•	Persistent long-term memory
	•	Reliability & guardrails

This is where agents become systems.